- Authors: Furkan Cantürk & Gemini 3 Pro
- Date: 21.11.2025

# Module 1: Foundations of Machine Learning & Classification

## 1. Introduction: The Fraud Detection Problem

In this crash course, we are tackling a classic problem: **Binary Classification**.
Specifically, we are looking at **Financial Risk/Fraud Detection**.

**The Scenario:** # You work for a bank. You have a database of customers and their financial history. Your goal is to build an automated system that flags a transaction or a customer as "Risk/Fraud" (1) or "Safe" (0).

**The Challenge - "The Needle in the Haystack":**
Fraud detection is difficult because:
1.  **Imbalance:** Most transactions are safe. Fraud is rare (often <1% of data).
2.  **Cost Asymmetry:** #     * **False Positive (Type I Error):** We flag a safe customer as fraud. They get annoyed (low cost).
    * **False Negative (Type II Error):** We let a fraudster through. The bank loses money (high cost).

**The Models We Will Build:**
We will progress from simple to complex:
1.  **Linear Regression:** (A naive approach to show why we need specialized classifiers).
2.  **Logistic Regression:** The industry standard baseline.
3.  **K-Nearest Neighbors (k-NN):** A distance-based approach.
4.  **Naive Bayes:** A probabilistic approach.
5.  **Decision Trees & Random Forests:** Non-linear, logic-based approaches.

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix
)

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Data Loading

We are using the **German Credit Risk dataset** (OpenML ID 31). 
* **Rows:** 1000
* **Columns:** 20 (7 numerical, 13 categorical)
* **Target:** `class` (Good vs Bad). 

*Note: In a real fraud scenario, you might have millions of rows. For this workshop, we use a smaller dataset to ensure our code runs in seconds, allowing us to focus on the concepts.*

In [14]:
# Load data directly from OpenML
print("Downloading dataset from OpenML...")
dataset = fetch_openml(data_id=31, as_frame=True, parser='auto')
df = dataset.frame

# Rename target for clarity
# Original: 'good', 'bad'. We map 'bad' to 1 (Positive/Risk) and 'good' to 0.
df['target'] = df['class'].map({'bad': 1, 'good': 0})
df = df.drop(columns=['class'])

print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (1000, 21)


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,target
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,0
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,22,none,own,1,skilled,1,none,yes,1
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,0
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,life insurance,45,none,for free,1,skilled,2,none,yes,0
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,no known property,53,none,for free,2,skilled,2,none,yes,1


## 3. Exploratory Data Analysis (EDA)

Before training models, we must understand our data. We look for:
1.  **Missing Values:** Do we need to impute data?
2.  **Data Types:** Which columns are numbers (needs scaling) vs strings (needs encoding)?
3.  **Class Balance:** Is this a "balanced" problem or an "imbalanced" one?

In [15]:
# 3.1 Check for missing values and data types
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nData Types:")
print(df.dtypes.value_counts())

# 3.2 Visualize Class Imbalance
# This is CRITICAL in fraud detection.
target_counts = df['target'].value_counts().reset_index()
target_counts.columns = ['Risk Status', 'Count']
target_counts['Label'] = target_counts['Risk Status'].map({0: 'Safe (0)', 1: 'Risk/Fraud (1)'})

fig = px.bar(target_counts, x='Label', y='Count', 
             title="Class Distribution: The Imbalance Problem",
             color='Label', 
             color_discrete_map={'Safe (0)': 'blue', 'Risk/Fraud (1)': 'red'})
fig.update_layout(height=400, width=600)
fig.show()

Missing values per column:
Series([], dtype: int64)

Data Types:
int64       7
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
category    1
Name: count, dtype: int64


### EDA Insight
You should see more "Safe" examples than "Risk" examples. 

**Why does this matter?**
If 90% of transactions are safe, a "dummy" model that **always predicts Safe** will have **90% Accuracy**. 
However, it will have **0% Recall** (it catches 0 fraud). 
This is why we will not rely solely on Accuracy.

## 4. Data Preprocessing & Pipelines

Machine Learning models are just math equations. They cannot understand the string "Check Account" or handle empty cells (NaN). We must translate data into math.

### The Concepts

1.  **Imputation:** #     * We cannot delete rows with missing data (data is expensive).
    * We fill missing numbers with the **Median** and missing categories with the **Most Frequent** value.

2.  **Encoding (Categorical -> Numerical):**
    * **One-Hot Encoding:** Creates a new binary column for every category (e.g., `Housing_Rent`, `Housing_Own`). Best for nominal data without order.
    * **Ordinal Encoding:** (Not used here, but good to know) converting "Low, Medium, High" to 1, 2, 3.

3.  **Scaling (Numerical -> Numerical):**
    * **Why?** A salary of 50,000 dominates an age of 30 in distance calculations (like k-NN).
    * **StandardScaler:** Transforms data to have Mean = 0 and Variance = 1. This is generally safer than MinMax scaling when outliers are present.

In [16]:
# Define which columns are which
X = df.drop(columns=['target'])
y = df['target']

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

# Create Transformers
# 1. Numerical Transformer: Impute -> Scale
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Categorical Transformer: Impute -> OneHotEncode
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

print("Preprocessing pipeline constructed.")

Preprocessing pipeline constructed.


## 5. Train / Test Split

**The Golden Rule of ML:** Never evaluate a model on data it has already seen.

We split data into:
* **Training Set (80%):** The model studies this.
* **Test Set (20%):** The final exam. The model never sees this until the very end.

**Stratification:**
Because fraud is rare, random splitting might put *all* the fraud cases in the Test set, leaving none for the model to learn from. `stratify=y` ensures the % of fraud is the same in both Train and Test sets.

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Data: {X_train.shape}")
print(f"Test Data: {X_test.shape}")

Training Data: (800, 20)
Test Data: (200, 20)


## 6. Model Training & Evaluation

We will loop through several models. For each model, we define a dictionary to store its performance.

### The Evaluation Metrics
* **Accuracy:** (TP + TN) / Total. *Dangerous for imbalance.*
* **Precision:** Of the ones we flagged as fraud, how many were actually fraud? (Low precision = annoying customers).
* **Recall (Sensitivity):** Of the actual fraud cases, how many did we catch? (Low recall = bank loses money).
* **F1-Score:** The harmonic mean of Precision and Recall. Good balance.
* **ROC-AUC:** Measures how good the model is at distinguishing classes, regardless of the threshold. 0.5 is random guessing, 1.0 is perfect.

In [18]:
# Helper Function for Evaluation
results_list = []

def evaluate_model(model_name, y_true, y_pred, y_proba=None):
    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    roc = "N/A"
    if y_proba is not None:
        roc = roc_auc_score(y_true, y_proba)
    
    print(f"--- {model_name} Results ---")
    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | AUC: {roc}")
    
    results_list.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": roc if roc != "N/A" else 0
    })

### Model 1: Logistic Regression (The Standard)

**Theory:** Despite the name, this is a **Classifier**.
Instead of fitting a straight line, it fits a **Sigmoid (S-Curve)**. This forces output to be between 0 and 1, which we interpret as "Probability of Fraud".

**Pros:** Interpretable, fast, works well on simple data.
**Cons:** Can struggle with complex, non-linear patterns.

In [19]:
log_reg = Pipeline(steps=[('preprocessor', preprocessor),
                          ('classifier', LogisticRegression(max_iter=1000))])

log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)
y_proba_log = log_reg.predict_proba(X_test)[:, 1]

evaluate_model("Logistic Regression", y_test, y_pred_log, y_proba_log)

--- Logistic Regression Results ---
Accuracy: 0.7800 | Precision: 0.6667 | Recall: 0.5333 | F1: 0.5926 | AUC: 0.8040476190476191


### Model 2: K-Nearest Neighbors (k-NN)

**Theory:** "Birds of a feather flock together."
To classify a new customer, the model looks at the `k` most similar customers in the training set. If the majority are fraud, the new one is fraud.

**Pros:** Simple logic, no training phase (lazy learner).
**Cons:** Very slow on large datasets (must measure distance to *everyone*), highly sensitive to outliers and bad scaling.

In [20]:
knn = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', KNeighborsClassifier(n_neighbors=5))])

knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
y_proba_knn = knn.predict_proba(X_test)[:, 1]

evaluate_model("k-NN (k=5)", y_test, y_pred_knn, y_proba_knn)

--- k-NN (k=5) Results ---
Accuracy: 0.7250 | Precision: 0.5676 | Recall: 0.3500 | F1: 0.4330 | AUC: 0.7294642857142857


### Model 3: Gaussian Naive Bayes

**Theory:** Based on Bayes' Theorem. It calculates the probability of features belonging to a class independently.
**Assumption:** It assumes features are **Independent** (e.g., your income has nothing to do with your age).

**Pros:** Extremely fast, works surprisingly well on small data.
**Cons:** The independence assumption is almost always wrong in real life.

In [21]:
nb = Pipeline(steps=[('preprocessor', preprocessor),
                     ('classifier', GaussianNB())])

nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
y_proba_nb = nb.predict_proba(X_test)[:, 1]

evaluate_model("Naive Bayes", y_test, y_pred_nb, y_proba_nb)

--- Naive Bayes Results ---
Accuracy: 0.6950 | Precision: 0.4937 | Recall: 0.6500 | F1: 0.5612 | AUC: 0.7410714285714286


### Model 4: Random Forest (The Strongest Baseline)

**Theory:** 
1.  **Decision Tree:** Asks a series of Yes/No questions to split data (e.g., "Is Debt > 10k?"). Prone to overfitting (memorizing data).
2.  **Random Forest:** Trains 100 separate trees on random subsets of data and averages their vote. This reduces overfitting.

**Pros:** High accuracy, handles non-linear data, robust.
**Cons:** Harder to interpret than a single tree or regression.

In [22]:
rf = Pipeline(steps=[('preprocessor', preprocessor),
                     ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

evaluate_model("Random Forest", y_test, y_pred_rf, y_proba_rf)

--- Random Forest Results ---
Accuracy: 0.7550 | Precision: 0.6486 | Recall: 0.4000 | F1: 0.4948 | AUC: 0.7619642857142858


## 7. Comparative Analysis & Visualization

Let's compare all models side-by-side.

In [23]:
results_df = pd.DataFrame(results_list)

# Plot F1 Score Comparison
fig = px.bar(results_df.round(3), x='Model', y='F1', 
             title="Model Comparison: F1 Score (Balance of Precision/Recall)",
             text='F1', color='F1', color_continuous_scale='Viridis')

fig.update_layout(width=700, height=500)
fig.show()

# Plot ROC-AUC Comparison
fig = px.bar(results_df.round(3), x='Model', y='AUC', 
             title="Model Comparison: ROC-AUC (Discriminative Power)",
             text='AUC', color='AUC', color_continuous_scale='Magma')

fig.update_layout(width=700, height=500)
fig.show()

### Confusion Matrix: Random Forest
Let's look closer at our best performer (likely Random Forest or Logistic Regression).

In [24]:
cm = confusion_matrix(y_test, y_pred_rf)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=['Predicted Safe', 'Predicted Fraud'],
    y=['Actual Safe', 'Actual Fraud'],
    colorscale='Reds',
    showscale=True
)
fig.update_layout(title="Confusion Matrix: Random Forest")
fig.update_layout(width=700, height=500)
fig.show()